# AutoVision: Phase 1 - Vehicle Feature Clustering & Unsupervised Discovery

Welcome to **AutoVision** (Bitirme Projesi).

### Mission Overview
- **Goal**: Discover latent clusters and evaluate unsupervised grouping of vehicle image features.
- **Dataset**: 5,000 samples across 10 vehicle models (500 images per class).
- **Features**: 512-dimensional deep feature representations (CNN embeddings).
- **Workflow**:
  1. Feature Exploration & L2 Normalization
  2. PCA Dimensionality Reduction & Variance Profiling
  3. Optimal Cluster Count Exploration (Elbow Method & Silhouette Analysis)
  4. Multi-Algorithm Clustering (K-Means, Agglomerative Ward/Cosine, GMM, MiniBatch K-Means)
  5. Evaluation Metrics (Unsupervised: Silhouette, Davies-Bouldin, Calinski-Harabasz; Supervised: ARI, NMI, Homogeneity, Completeness, Purity)
  6. 2D t-SNE Visualization & Ground Truth Comparison
  7. Contingency Matrix & Visual Error Analysis using `train_images/`

## 1. Imports & Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

from sklearn.preprocessing import StandardScaler, normalize
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans, MiniBatchKMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn import metrics

# Set seeds for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

print("Libraries loaded successfully.")

## 2. Load Data & Pre-extracted Embeddings

In [ ]:
df_data = pd.read_csv('train_data.csv', header=None, names=['label', 'filename'])
X = pd.read_csv('train_features.csv', header=None).values
y = df_data['label'].values
unique_classes = np.unique(y)

print(f"Samples: {X.shape[0]}")
print(f"Feature dimension: {X.shape[1]}")
print(f"Distinct vehicle classes: {len(unique_classes)}")
print("\nClass counts:")
display(df_data['label'].value_counts())

## 3. Feature Distribution & Normalization

Deep CNN embeddings encode directional semantic similarities in high-dimensional space. Therefore, **L2 Normalization** (unit hypersphere projection) makes Euclidean distance equivalent to Cosine distance, which is standard for clustering image representations.

In [ ]:
# Normalize features
X_l2 = normalize(X, norm='l2')
X_std = StandardScaler().fit_transform(X)

raw_norms = np.linalg.norm(X, axis=1)
l2_norms = np.linalg.norm(X_l2, axis=1)

print(f"Raw norms: min={raw_norms.min():.2f}, max={raw_norms.max():.2f}, mean={raw_norms.mean():.2f}")
print(f"L2 normalized norms: min={l2_norms.min():.2f}, max={l2_norms.max():.2f}")

## 4. PCA Variance Profiling

In [ ]:
pca = PCA().fit(X)
cum_var = np.cumsum(pca.explained_variance_ratio_)

n_80 = np.argmax(cum_var >= 0.80) + 1
n_90 = np.argmax(cum_var >= 0.90) + 1
n_95 = np.argmax(cum_var >= 0.95) + 1

print(f"80% variance captured by: {n_80} components")
print(f"90% variance captured by: {n_90} components")
print(f"95% variance captured by: {n_95} components")

plt.figure(figsize=(9, 4))
plt.plot(range(1, len(cum_var) + 1), cum_var, color='teal', linewidth=2)
plt.axhline(y=0.80, color='grey', linestyle=':', label=f'80% ({n_80} PCs)')
plt.axhline(y=0.90, color='crimson', linestyle='--', label=f'90% ({n_90} PCs)')
plt.axhline(y=0.95, color='darkorange', linestyle='-.', label=f'95% ({n_95} PCs)')
plt.title('PCA Cumulative Explained Variance', fontsize=12, fontweight='bold')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Variance Ratio')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

## 5. Finding Optimal $k$ (Elbow Method & Silhouette Analysis)

In [ ]:
k_range = list(range(2, 21))
inertias = []
sil_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=5)
    preds = km.fit_predict(X_l2)
    inertias.append(km.inertia_)
    sil_scores.append(metrics.silhouette_score(X_l2, preds))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 4))
ax1.plot(k_range, inertias, marker='o', color='royalblue', linewidth=2)
ax1.set_title('Elbow Curve (Inertia / WCSS)', fontweight='bold')
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Inertia')
ax1.axvline(x=10, color='crimson', linestyle='--', label='Ground Truth k=10')
ax1.legend()
ax1.grid(True, linestyle='--', alpha=0.6)

ax2.plot(k_range, sil_scores, marker='s', color='darkorange', linewidth=2)
ax2.set_title('Silhouette Score vs k', fontweight='bold')
ax2.set_xlabel('Number of Clusters (k)')
ax2.set_ylabel('Silhouette Score')
ax2.axvline(x=10, color='crimson', linestyle='--', label='Ground Truth k=10')
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

## 6. Clustering Models & Metric Evaluation ($k=10$)

In [ ]:
def calculate_purity(y_true, y_pred):
    contingency = metrics.cluster.contingency_matrix(y_true, y_pred)
    return np.sum(np.amax(contingency, axis=0)) / np.sum(contingency)

def evaluate(X_data, y_true, y_pred, name):
    sil = metrics.silhouette_score(X_data, y_pred)
    ch = metrics.calinski_harabasz_score(X_data, y_pred)
    db = metrics.davies_bouldin_score(X_data, y_pred)
    ari = metrics.adjusted_rand_score(y_true, y_pred)
    nmi = metrics.normalized_mutual_info_score(y_true, y_pred)
    homo, comp, vm = metrics.homogeneity_completeness_v_measure(y_true, y_pred)
    purity = calculate_purity(y_true, y_pred)
    return {
        'Algorithm': name,
        'Silhouette': round(sil, 4),
        'Calinski-Harabasz': round(ch, 2),
        'Davies-Bouldin': round(db, 4),
        'ARI': round(ari, 4),
        'NMI': round(nmi, 4),
        'Homogeneity': round(homo, 4),
        'Completeness': round(comp, 4),
        'V-Measure': round(vm, 4),
        'Purity': round(purity, 4)
    }

# 1. K-Means (L2 normalized)
km = KMeans(n_clusters=10, random_state=RANDOM_STATE, n_init=10)
y_km = km.fit_predict(X_l2)

# 2. MiniBatch K-Means
mb_km = MiniBatchKMeans(n_clusters=10, random_state=RANDOM_STATE, batch_size=256, n_init=10)
y_mb = mb_km.fit_predict(X_l2)

# 3. Agglomerative (Ward linkage)
agg_ward = AgglomerativeClustering(n_clusters=10, linkage='ward')
y_agg_ward = agg_ward.fit_predict(X_l2)

# 4. Agglomerative (Average linkage with Cosine)
agg_avg = AgglomerativeClustering(n_clusters=10, metric='cosine', linkage='average')
y_agg_avg = agg_avg.fit_predict(X)

# 5. Gaussian Mixture Model (GMM)
gmm = GaussianMixture(n_components=10, covariance_type='diag', random_state=RANDOM_STATE)
y_gmm = gmm.fit_predict(X_l2)

# Compile metrics table
eval_results = [
    evaluate(X_l2, y, y_km, 'K-Means (L2-norm)'),
    evaluate(X_l2, y, y_mb, 'MiniBatch K-Means'),
    evaluate(X_l2, y, y_agg_ward, 'Agglomerative (Ward)'),
    evaluate(X, y, y_agg_avg, 'Agglomerative (Cosine-Avg)'),
    evaluate(X_l2, y, y_gmm, 'Gaussian Mixture (GMM)')
]

df_eval = pd.DataFrame(eval_results)
display(df_eval.sort_values(by='ARI', ascending=False))

## 7. 2D t-SNE Visualization: Ground Truth vs Clusters

In [ ]:
print("Computing t-SNE embedding...")
tsne = TSNE(n_components=2, perplexity=35, random_state=RANDOM_STATE, max_iter=1000)
X_tsne = tsne.fit_transform(X_l2)

fig, axes = plt.subplots(2, 2, figsize=(18, 15))

def plot_cluster_scatter(ax, coords, labels_arr, title, legend_title):
    unique_vals = np.unique(labels_arr)
    palette = sns.color_palette('tab10' if len(unique_vals) <= 10 else 'tab20', len(unique_vals))
    for i, val in enumerate(unique_vals):
        mask = labels_arr == val
        lbl = f'Cluster {val}' if isinstance(val, (int, np.integer)) else val.replace('_', ' ')
        ax.scatter(coords[mask, 0], coords[mask, 1], label=lbl, s=14, alpha=0.7, color=palette[i])
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')
    ax.legend(title=legend_title, bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
    ax.grid(True, linestyle='--', alpha=0.4)

plot_cluster_scatter(axes[0, 0], X_tsne, y, 'Ground Truth Vehicle Classes', 'Ground Truth')
plot_cluster_scatter(axes[0, 1], X_tsne, y_km, 'K-Means Clusters (L2-norm)', 'Cluster')
plot_cluster_scatter(axes[1, 0], X_tsne, y_agg_ward, 'Agglomerative (Ward Linkage)', 'Cluster')
plot_cluster_scatter(axes[1, 1], X_tsne, y_gmm, 'Gaussian Mixture Model (GMM)', 'Component')

plt.tight_layout()
plt.show()

## 8. Contingency Matrix (Ground Truth vs K-Means)

In [ ]:
contingency = metrics.cluster.contingency_matrix(y, y_km)
df_cont = pd.DataFrame(
    contingency,
    index=[c.replace('_', ' ') for c in unique_classes],
    columns=[f'Cluster {i}' for i in range(10)]
)

plt.figure(figsize=(11, 7))
sns.heatmap(df_cont, annot=True, fmt='d', cmap='Blues', cbar=True)
plt.title('Contingency Matrix: Ground Truth vs K-Means Clusters', fontsize=12, fontweight='bold')
plt.xlabel('Assigned Cluster ID')
plt.ylabel('True Vehicle Model')
plt.xticks(rotation=45, ha='right')
plt.show()

## 9. Visual Cluster Inspection (Displaying Images from Clusters)

In [ ]:
# Display 5 sample vehicle images from Cluster 0 to verify visual coherence
target_cluster = 0
cluster_indices = np.where(y_km == target_cluster)[0][:5]

fig, axes = plt.subplots(1, len(cluster_indices), figsize=(16, 3))
for ax, idx in zip(axes, cluster_indices):
    row = df_data.iloc[idx]
    img_path = os.path.join('train_images', row['label'], row['filename'])
    if os.path.exists(img_path):
        img = Image.open(img_path)
        ax.imshow(img)
    ax.set_title(f"{row['label']}", fontsize=8)
    ax.axis('off')

plt.suptitle(f"Sample Images assigned to Cluster {target_cluster}", fontsize=12, fontweight='bold')
plt.show()